<center><font size=6>Agent2Agent Protocol</center></font>

# 1. What is A2A (Agent2Agent Protocol)?

## The Problem

In the previous notebooks we saw how **MCP (Model Context Protocol)** lets an LLM call external tools, and how **ACP (Agent Communication Protocol)** lets agents talk to each other. Both are useful — but the industry needed a **single, vendor-neutral standard** for agent-to-agent communication backed by broad ecosystem support.

## The Solution: A2A

**A2A (Agent2Agent)** is an open protocol for agent-to-agent communication, originally created by Google and now a **Linux Foundation standard** with 50+ partners including AWS, Microsoft, Salesforce, and SAP. Think of it as the **HTTP of AI agents** — a universal wire protocol that lets any agent discover and communicate with any other agent.

### Architecture

```
┌──────────────┐     A2A/HTTP      ┌──────────────┐     A2A/HTTP     ┌──────────────┐
│   Notebook   │ ───────────────►  │   Advisor    │ ────────────────► │   Data       │ ──► Real APIs
│   (Client)   │   port 8001       │    Agent     │   port 8000      │   Agents     │
└──────────────┘                   └──────────────┘                   └──────────────┘
```

### How A2A Compares

| | **MCP** | **ACP** | **A2A** |
|---|---------|---------|--------|
| **Direction** | Vertical: LLM → Tools | Horizontal: Agent → Agent | Horizontal: Agent → Agent |
| **Discovery** | Tool schemas | `GET /agents` | **Agent Card** at `/.well-known/agent.json` |
| **Transport** | stdio / SSE | HTTP | **JSON-RPC 2.0** over HTTP |
| **Capabilities** | Tool descriptions | Agent manifests | **Skills** on Agent Card |
| **Governance** | Anthropic | BeeAI (open-source) | **Linux Foundation** (50+ partners) |
| **Best for** | Giving one LLM access to data & actions | Composing agents in Python | **Industry-standard** agent interoperability |

### Key A2A Concepts

- **Agent Card** — A JSON document (served at `/.well-known/agent.json`) that describes what an agent can do, similar to an OpenAPI spec for agents
- **Skills** — Capabilities listed on the Agent Card, telling clients what the agent is good at
- **AgentExecutor** — The server-side pattern: implement `execute()` to handle requests and return results via an event queue
- **JSON-RPC 2.0** — The wire protocol: `message/send` to send messages, `tasks/get` to check status

In this notebook, we'll build the **same travel research** use case from the MCP and ACP notebooks — but this time using the A2A protocol.

# 2. Setup

## Install Dependencies

In [1]:
!pip install "a2a-sdk[http-server]" langchain-openai httpx uvicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 4.4 MB/s eta 0:00:00


**Note:**
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells starting from the next cell onward.

## Imports

In [3]:
import json
import os
import subprocess
import sys
import time
import asyncio

from pathlib import Path
from getpass import getpass


def load_openai_config(required_for: str) -> dict:
    config_path = Path("config.json")
    config = {}
    if config_path.exists():
        config = json.loads(config_path.read_text())

    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    def read_value(key: str, default: str | None = None) -> str | None:
        if config.get(key):
            return config[key]
        env_value = os.environ.get(key)
        if env_value:
            return env_value
        if userdata is not None:
            try:
                secret_value = userdata.get(key)
            except Exception:
                secret_value = None
            if secret_value:
                return secret_value
        return default

    openai_api_key = read_value("OPENAI_API_KEY")
    openai_api_base = read_value("OPENAI_API_BASE", "https://api.openai.com/v1")

    if not openai_api_key:
        openai_api_key = getpass("Enter your OPENAI_API_KEY: ").strip()

    if not openai_api_key:
        raise ValueError(f"OPENAI_API_KEY is required to run {required_for}.")

    config = {
        "OPENAI_API_KEY": openai_api_key,
        "OPENAI_API_BASE": openai_api_base,
    }
    config_path.write_text(json.dumps(config, indent=2))

    print(f"Saved OpenAI config to {config_path.resolve()}")
    print(f"Using OPENAI_API_BASE={openai_api_base}")
    return config

config = load_openai_config("the A2A advisor agent")

Saved OpenAI config to /content/config.json


In [4]:
import nest_asyncio
nest_asyncio.apply()

# 3. Building the Data Agent

Our A2A data agent hosts **three skills** on a single server, each wrapping one free public API:

| Skill | API | What it does |
|-------|-----|-------------|
| `weather` | Open-Meteo | Geocodes a city, returns current weather + 7-day forecast |
| `country` | REST Countries | Returns country profile (capital, languages, currency) |
| `holiday` | Nager.Date | Lists public holidays for a country code + year |

### A2A Server Pattern

Every A2A server needs three things:

1. **Agent Card** — describes the agent and its skills (served at `/.well-known/agent.json`)
2. **AgentExecutor** — implements `execute()` to handle incoming messages
3. **A2AStarletteApplication** — wires the card and executor into an HTTP server

The executor reads user input via `context.get_user_input()` and returns results by enqueuing a message onto the `event_queue`. Simple routing by message prefix (`weather:`, `country:`, `holiday:`) lets one executor serve all three skills.

In [5]:
%%writefile data_agent_a2a.py
# ============================================================
# A2A Data Agent Server
# Three specialized skills wrapping free public APIs
# ============================================================

import httpx
import uvicorn

from a2a.types import AgentCard, AgentSkill, AgentCapabilities
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_agent_text_message


# ── Skills ────────────────────────────────────────────────────

weather_skill = AgentSkill(
    id="weather",
    name="Weather Lookup",
    description="Get current weather and 7-day forecast for a city. Query format: 'weather: <city>'",
    tags=["weather", "forecast", "temperature"],
    examples=["weather: Tokyo", "weather: Paris"],
)

country_skill = AgentSkill(
    id="country",
    name="Country Information",
    description="Get country profile including capital, population, languages, and currency. Query format: 'country: <country name>'",
    tags=["country", "geography", "travel"],
    examples=["country: Japan", "country: France"],
)

holiday_skill = AgentSkill(
    id="holiday",
    name="Public Holidays",
    description="Get public holidays for a country and year. Query format: 'holiday: <2-letter code> <year>'",
    tags=["holidays", "calendar", "travel"],
    examples=["holiday: JP 2026", "holiday: US 2026"],
)


# ── Agent Card ────────────────────────────────────────────────

agent_card = AgentCard(
    name="Travel Data Agent",
    description=(
        "Provides weather forecasts, country profiles, and public holidays. "
        "Prefix your message with the skill name: weather: / country: / holiday:"
    ),
    url="http://localhost:8000/",
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[weather_skill, country_skill, holiday_skill],
)


# ── API Helpers ───────────────────────────────────────────────

async def _geocode(city: str) -> dict | None:
    """Look up latitude/longitude for a city name."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
        )
        data = resp.json()
        if "results" not in data:
            return None
        r = data["results"][0]
        return {
            "name": r["name"],
            "country": r.get("country", "Unknown"),
            "lat": r["latitude"],
            "lon": r["longitude"],
        }


async def _get_weather(city: str) -> str:
    geo = await _geocode(city)
    if not geo:
        return f"Could not find city: {city}"

    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": geo["lat"],
                "longitude": geo["lon"],
                "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
                "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
                "timezone": "auto",
                "forecast_days": 7,
            },
        )
        data = resp.json()

    c = data["current"]
    d = data["daily"]

    report = f"Weather Report: {geo['name']}, {geo['country']}\n"
    report += "=" * 50 + "\n\n"
    report += "Current Conditions:\n"
    report += f"  Temperature: {c['temperature_2m']}\u00b0C\n"
    report += f"  Humidity:    {c['relative_humidity_2m']}%\n"
    report += f"  Wind Speed:  {c['wind_speed_10m']} km/h\n\n"
    report += "7-Day Forecast:\n"
    for i in range(7):
        report += (
            f"  {d['time'][i]}: "
            f"{d['temperature_2m_min'][i]}\u00b0C \u2013 {d['temperature_2m_max'][i]}\u00b0C, "
            f"rain {d['precipitation_sum'][i]} mm\n"
        )
    return report


async def _get_country(name: str) -> str:
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"https://restcountries.com/v3.1/name/{name}",
            params={"fullText": "false"},
        )
        if resp.status_code != 200:
            return f"Country not found: {name}"
        countries = resp.json()

    c = countries[0]
    languages = ", ".join(c.get("languages", {}).values()) or "N/A"
    currencies = ", ".join(
        f"{v['name']} ({v.get('symbol', '')})"
        for v in c.get("currencies", {}).values()
    ) or "N/A"
    capitals = ", ".join(c.get("capital", [])) or "N/A"

    report = f"Country Profile: {c['name']['common']}\n"
    report += "=" * 50 + "\n\n"
    report += f"  Official Name: {c['name'].get('official', 'N/A')}\n"
    report += f"  Capital:       {capitals}\n"
    report += f"  Region:        {c.get('region', 'N/A')} / {c.get('subregion', 'N/A')}\n"
    report += f"  Population:    {c.get('population', 0):,}\n"
    report += f"  Languages:     {languages}\n"
    report += f"  Currencies:    {currencies}\n"
    report += f"  Timezone(s):   {', '.join(c.get('timezones', []))}\n"
    report += f"  Drives on:     {c.get('car', {}).get('side', 'N/A')} side\n"
    return report


async def _get_holidays(text: str) -> str:
    parts = text.strip().split()
    if len(parts) < 2:
        return "Please provide a country code and year, e.g. 'JP 2026'."

    country_code, year = parts[0].upper(), parts[1]

    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}"
        )
        if resp.status_code != 200:
            return f"Could not fetch holidays for {country_code} in {year}."
        holidays = resp.json()

    report = f"Public Holidays: {country_code} ({year})\n"
    report += "=" * 50 + "\n\n"
    for h in holidays:
        report += f"  {h['date']}  {h['localName']} ({h['name']})\n"
    report += f"\nTotal: {len(holidays)} public holidays\n"
    return report


# ── Agent Executor ────────────────────────────────────────────

class DataAgentExecutor(AgentExecutor):

    async def execute(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        user_input = context.get_user_input().strip()

        if user_input.lower().startswith("weather:"):
            result = await _get_weather(user_input[8:].strip())
        elif user_input.lower().startswith("country:"):
            result = await _get_country(user_input[8:].strip())
        elif user_input.lower().startswith("holiday:"):
            result = await _get_holidays(user_input[8:].strip())
        else:
            result = (
                f"Unknown request: '{user_input}'\n"
                "Prefix your query with: weather: / country: / holiday:"
            )

        await event_queue.enqueue_event(new_agent_text_message(result))

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        raise Exception("cancel not supported")


# ── Server Wiring ─────────────────────────────────────────────

request_handler = DefaultRequestHandler(
    agent_executor=DataAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

server_app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
)

if __name__ == "__main__":
    uvicorn.run(server_app.build(), host="0.0.0.0", port=8000)

Writing data_agent_a2a.py


# 4. Building the Advisor Agent

The Advisor is an **LLM-powered agent** that acts as an intelligent orchestrator. When it receives a user query, it:

1. **Discovers** the data agent's skills by fetching its Agent Card via `GET /.well-known/agent.json`
2. **Plans** which skills to invoke using the LLM (outputs a JSON plan)
3. **Executes** the plan by sending A2A messages (JSON-RPC `message/send`) to the data agent
4. **Synthesizes** all collected data into a final advisory using the LLM

The Advisor never calls any API directly — it **delegates** to a peer agent via A2A.

In [6]:
%%writefile advisor_agent_a2a.py
# ============================================================
# A2A Advisor Agent Server
# LLM-powered orchestrator that delegates to data agents
# ============================================================

import asyncio
import json
from uuid import uuid4

import httpx
import uvicorn
from langchain_openai import ChatOpenAI

from a2a.types import AgentCard, AgentSkill, AgentCapabilities
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils import new_agent_text_message


# ── Configuration ─────────────────────────────────────────────

with open("config.json") as f:
    config = json.load(f)

llm = ChatOpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config.get("OPENAI_API_BASE") or "https://api.openai.com/v1",
    model="gpt-4o-mini",
    temperature=0,
)

DATA_AGENT_URL = "http://localhost:8000"


# ── Agent Card ────────────────────────────────────────────────

agent_card = AgentCard(
    name="Travel Advisor Agent",
    description=(
        "LLM-powered travel advisor that discovers data agents at runtime, "
        "plans which to consult, and synthesizes results into a travel advisory."
    ),
    url="http://localhost:8001/",
    version="1.0.0",
    default_input_modes=["text"],
    default_output_modes=["text"],
    capabilities=AgentCapabilities(streaming=False),
    skills=[
        AgentSkill(
            id="travel_advisory",
            name="Travel Advisory",
            description="Answer any travel question by consulting specialized data agents",
            tags=["travel", "advisory", "planning"],
            examples=[
                "What's the weather in Tokyo?",
                "Tell me about France",
                "Complete travel brief for Japan",
            ],
        )
    ],
)


# ── A2A Protocol Helpers ──────────────────────────────────────

def _extract_text_from_a2a_result(result: dict) -> str:
    """Extract text from an A2A JSON-RPC response (Task or Message)."""
    texts = []

    # Task response: check status.message and artifacts
    if "status" in result:
        msg = result.get("status", {}).get("message")
        if msg:
            for part in msg.get("parts", []):
                if part.get("kind") == "text":
                    texts.append(part["text"])
        for artifact in result.get("artifacts", []):
            for part in artifact.get("parts", []):
                if part.get("kind") == "text":
                    texts.append(part["text"])

    # Direct Message response: check parts
    elif "parts" in result:
        for part in result.get("parts", []):
            if part.get("kind") == "text":
                texts.append(part["text"])

    return "\n".join(texts) if texts else str(result)


async def a2a_send_message(
    http_client: httpx.AsyncClient, base_url: str, text: str
) -> str:
    """Send a message via A2A JSON-RPC and return the response text."""
    request = {
        "jsonrpc": "2.0",
        "id": uuid4().hex,
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": text}],
                "messageId": uuid4().hex,
            }
        },
    }
    resp = await http_client.post(f"{base_url}/", json=request)
    resp.raise_for_status()
    result = resp.json().get("result", {})
    return _extract_text_from_a2a_result(result)


async def a2a_discover_skills(
    http_client: httpx.AsyncClient, base_url: str
) -> list[dict]:
    """Fetch the Agent Card and return the skills list."""
    resp = await http_client.get(f"{base_url}/.well-known/agent.json")
    resp.raise_for_status()
    card = resp.json()
    return card.get("skills", [])


# ── Advisor Executor ──────────────────────────────────────────

class AdvisorAgentExecutor(AgentExecutor):

    async def execute(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        user_query = context.get_user_input()

        async with httpx.AsyncClient(timeout=60) as http_client:
            # Step 1: Discover data agent skills via Agent Card
            skills = await a2a_discover_skills(http_client, DATA_AGENT_URL)
            skills_desc = "\n".join(
                f"- {s['id']}: {s['description']} "
                f"(examples: {', '.join(s.get('examples', []))})"
                for s in skills
            )

            # Step 2: LLM planning — decide which skills to invoke
            planning_prompt = f"""You are a travel advisor. A user has asked:

\"{user_query}\"

You can consult these data agent skills (via A2A protocol):
{skills_desc}

Decide which skills to call and what query to send each.
Respond with ONLY a JSON object:
{{
  "calls": [
    {{"skill": "skill_id", "query": "the query text after the prefix"}}
  ]
}}

Rules:
- For weather, query is the city name (e.g. "Tokyo")
- For country, query is the country name (e.g. "Japan")
- For holiday, query is "CC YYYY" (e.g. "JP 2026")
- You may call the same skill multiple times with different queries
- Include all skills needed to fully answer the user's question"""

            plan_response = await llm.ainvoke(planning_prompt)
            plan_text = plan_response.content

            # Parse the JSON plan
            try:
                if "```" in plan_text:
                    plan_text = plan_text.split("```")[1]
                    if plan_text.startswith("json"):
                        plan_text = plan_text[4:]
                plan = json.loads(plan_text.strip())
            except (json.JSONDecodeError, IndexError):
                await event_queue.enqueue_event(
                    new_agent_text_message(
                        f"Failed to parse LLM plan. Raw output:\n{plan_response.content}"
                    )
                )
                return

            # Step 3: Execute the plan — call data agent for each skill
            calls = plan.get("calls", [])
            if not calls:
                await event_queue.enqueue_event(
                    new_agent_text_message("No data agents needed for this query.")
                )
                return

            async def _call(call: dict) -> str:
                query = f"{call['skill']}: {call['query']}"
                return await a2a_send_message(http_client, DATA_AGENT_URL, query)

            results = await asyncio.gather(
                *[_call(c) for c in calls], return_exceptions=True
            )

            collected = ""
            for call, result in zip(calls, results):
                collected += f"\n--- {call['skill']} (query: {call['query']}) ---\n"
                if isinstance(result, Exception):
                    collected += f"Error: {result}\n"
                else:
                    collected += f"{result}\n"

            # Step 4: LLM synthesis — combine results into advisory
            synthesis_prompt = f"""You are a travel advisor. The user asked:

\"{user_query}\"

You consulted data agents via the A2A protocol and received:
{collected}

Write a helpful, well-structured travel advisory that synthesizes the information.
Be concise but informative."""

            final_response = await llm.ainvoke(synthesis_prompt)
            await event_queue.enqueue_event(
                new_agent_text_message(final_response.content)
            )

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        raise Exception("cancel not supported")


# ── Server Wiring ─────────────────────────────────────────────

request_handler = DefaultRequestHandler(
    agent_executor=AdvisorAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

server_app = A2AStarletteApplication(
    agent_card=agent_card,
    http_handler=request_handler,
)

if __name__ == "__main__":
    uvicorn.run(server_app.build(), host="0.0.0.0", port=8001)

Writing advisor_agent_a2a.py


# 5. Understanding the Architecture

Let's review what we just built:

### Components

| Component | File | Port | LLM? | Purpose |
|-----------|------|------|------|---------|
| Data Agent | `data_agent_a2a.py` | 8000 | No | Wraps free APIs as A2A skills |
| Advisor Agent | `advisor_agent_a2a.py` | 8001 | Yes | Plans, delegates, and synthesizes |
| Client | This notebook | — | No | Sends queries, displays results |

### Agent Card = The A2A Discovery Mechanism

Every A2A agent serves a **JSON document** at `/.well-known/agent.json` describing itself:

```json
{
  "name": "Travel Data Agent",
  "description": "Provides weather, country info, and holidays",
  "skills": [
    {"id": "weather", "name": "Weather Lookup", ...},
    {"id": "country", "name": "Country Information", ...},
    {"id": "holiday", "name": "Public Holidays", ...}
  ]
}
```

This is the A2A equivalent of MCP tool schemas and ACP agent manifests. The Advisor fetches this card at runtime to discover what skills are available — **no hardcoding required**.

### Communication Flow

```
1. Client sends "Tell me about Japan" to Advisor via JSON-RPC (port 8001)
2. Advisor fetches Agent Card from Data Agent (GET /.well-known/agent.json on port 8000)
3. Advisor's LLM plans: call country("Japan"), weather("Tokyo"), holiday("JP 2026")
4. Advisor sends three A2A messages to Data Agent in parallel (JSON-RPC message/send)
5. Data Agent routes each by prefix and fetches from the appropriate API
6. Advisor's LLM synthesizes all results into a travel advisory
7. Client receives the final response
```

### How This Differs from MCP and ACP

| Aspect | MCP Notebook | ACP Notebook | A2A Notebook |
|--------|-------------|-------------|-------------|
| Discovery | Client reads tool schemas | `GET /agents` | `GET /.well-known/agent.json` |
| Wire protocol | stdio / SSE | REST HTTP | **JSON-RPC 2.0** |
| Server pattern | Tool functions | `@server.agent` decorator | **AgentExecutor** class |
| Governance | Anthropic | BeeAI | **Linux Foundation** |
| Same use case? | Yes | Yes | Yes |

# 6. Starting the Agents

In [7]:
# Kill any stale processes on ports 8000 and 8001
!kill $(lsof -t -i:8000) 2>/dev/null; true
!kill $(lsof -t -i:8001) 2>/dev/null; true

def ensure_running(proc, name):
    if proc.poll() is not None:
        stdout, stderr = proc.communicate(timeout=1)
        raise RuntimeError(
            f"{name} exited early.\n\nSTDOUT:\n{stdout}\n\nSTDERR:\n{stderr}"
        )

# Launch both agent servers as background subprocesses
data_proc = subprocess.Popen(
    [sys.executable, "data_agent_a2a.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
print(f"Data agent server started (PID: {data_proc.pid})")

# Give the data agent a moment to start before launching the advisor
time.sleep(3)
ensure_running(data_proc, "Data agent server")

advisor_proc = subprocess.Popen(
    [sys.executable, "advisor_agent_a2a.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
print(f"Advisor agent server started (PID: {advisor_proc.pid})")

# Wait for both servers to be ready
time.sleep(5)
ensure_running(data_proc, "Data agent server")
ensure_running(advisor_proc, "Advisor agent server")
print("Both servers are ready.")

Data agent server started (PID: 1332)
Advisor agent server started (PID: 1345)
Both servers are ready.


In [10]:
# Verify both agents by fetching their Agent Cards
import httpx
import asyncio
import time

async def verify_servers(max_retries=5):
    async with httpx.AsyncClient() as client:
        for i in range(max_retries):
            try:
                print(f"Attempt {i+1}: Checking Data Agent (port 8000)...")
                resp_data = await client.get("http://localhost:8000/.well-known/agent.json", timeout=5.0)
                card_data = resp_data.json()

                print(f"Attempt {i+1}: Checking Advisor Agent (port 8001)...")
                resp_adv = await client.get("http://localhost:8001/.well-known/agent.json", timeout=5.0)
                card_adv = resp_adv.json()

                print("\nData Agent (port 8000) — Agent Card:")
                print(f"  Name: {card_data['name']}")
                print(f"  Skills: {[s['id'] for s in card_data.get('skills', [])]}")

                print("\nAdvisor Agent (port 8001) — Agent Card:")
                print(f"  Name: {card_adv['name']}")
                print(f"  Skills: {[s['id'] for s in card_adv.get('skills', [])]}")
                return
            except (httpx.ConnectError, httpx.TimeoutException):
                if i < max_retries - 1:
                    print("Servers not ready yet. Retrying in 2 seconds...")
                    await asyncio.sleep(2)
                else:
                    print("\nError: Could not connect to servers after multiple attempts. Check background logs.")
                    raise

# Using the existing event loop via asyncio.get_event_loop().run_until_complete()
# or simply awaiting if in a top-level await environment like Colab.
await verify_servers()

Attempt 1: Checking Data Agent (port 8000)...
Attempt 1: Checking Advisor Agent (port 8001)...

Data Agent (port 8000) — Agent Card:
  Name: Travel Data Agent
  Skills: ['weather', 'country', 'holiday']

Advisor Agent (port 8001) — Agent Card:
  Name: Travel Advisor Agent
  Skills: ['travel_advisory']


# 7. Client Helper

The notebook client sends a **JSON-RPC 2.0** request to the Advisor agent. This is the native A2A wire format — every A2A message is a JSON-RPC call to `message/send`.

In [11]:
from uuid import uuid4
import httpx


def _extract_text_from_a2a_result(result: dict) -> str:
    """Extract text from an A2A JSON-RPC response (Task or Message)."""
    texts = []

    # Task response: check status.message and artifacts
    if "status" in result:
        msg = result.get("status", {}).get("message")
        if msg:
            for part in msg.get("parts", []):
                if part.get("kind") == "text":
                    texts.append(part["text"])
        for artifact in result.get("artifacts", []):
            for part in artifact.get("parts", []):
                if part.get("kind") == "text":
                    texts.append(part["text"])

    # Direct Message response: check parts
    elif "parts" in result:
        for part in result.get("parts", []):
            if part.get("kind") == "text":
                texts.append(part["text"])

    return "\n".join(texts) if texts else str(result)


async def run_query(user_query: str) -> str:
    """Send a query to the Advisor agent via A2A JSON-RPC."""
    print(f"Query: {user_query}")
    print("=" * 60)

    request = {
        "jsonrpc": "2.0",
        "id": uuid4().hex,
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": user_query}],
                "messageId": uuid4().hex,
            }
        },
    }

    async with httpx.AsyncClient(timeout=120) as client:
        resp = await client.post("http://localhost:8001/", json=request)
        resp.raise_for_status()

    result = resp.json().get("result", {})
    return _extract_text_from_a2a_result(result)

# 8. Demo: Single Skill — Weather

The simplest case: the Advisor's LLM determines that only the `weather` skill is needed.

In [12]:
response = await run_query("What's the weather in Tokyo?")
print(response)

Query: What's the weather in Tokyo?
### Travel Advisory: Weather in Tokyo, Japan

**Current Weather Conditions:**
- **Temperature:** 13.0°C
- **Humidity:** 45%
- **Wind Speed:** 3.5 km/h

**7-Day Weather Forecast:**
- **March 15:** 4.9°C – 13.9°C, no rain
- **March 16:** 6.6°C – 12.7°C, no rain
- **March 17:** 4.1°C – 14.1°C, no rain
- **March 18:** 5.2°C – 14.7°C, no rain
- **March 19:** 8.7°C – 15.0°C, light rain (1.1 mm)
- **March 20:** 5.8°C – 18.0°C, light rain (0.4 mm)
- **March 21:** 3.9°C – 15.8°C, no rain

**General Information:**
- **Country:** Japan
- **Capital:** Tokyo
- **Population:** Approximately 123.2 million
- **Official Language:** Japanese
- **Currency:** Japanese yen (¥)
- **Timezone:** UTC+09:00
- **Driving:** Left side of the road

**Travel Tips:**
- Dress in layers to accommodate the cool temperatures, especially in the mornings and evenings.
- Carry an umbrella or raincoat for the forecasted light rain on March 19 and 20.
- Enjoy the vibrant culture and attract

# 9. Demo: Two Skills — Country + Weather

Now the Advisor must call **two** skills — `country` for languages and currency, and `weather` for Paris conditions. Watch how it plans both calls and synthesizes the results.

In [13]:
response = await run_query(
    "Tell me about France — what languages do they speak, "
    "what currency do they use, and what's the weather like in Paris?"
)
print(response)

Query: Tell me about France — what languages do they speak, what currency do they use, and what's the weather like in Paris?
### Travel Advisory: France

**Country Overview:**
- **Official Name:** French Republic
- **Capital:** Paris
- **Region:** Western Europe
- **Population:** Approximately 66.4 million

**Language:**
- The official language spoken in France is **French**. While many people in urban areas and tourist destinations may speak English, it's beneficial to learn a few basic French phrases to enhance your experience.

**Currency:**
- The currency used in France is the **euro (€)**. Credit cards are widely accepted, but it's advisable to carry some cash for smaller establishments.

**Weather in Paris:**
- **Current Conditions:** The temperature is around **3.2°C** with high humidity (86%) and light winds (1.8 km/h).
  
**7-Day Weather Forecast:**
- **March 15:** 2.0°C – 12.7°C, no rain
- **March 16:** 7.5°C – 13.2°C, no rain
- **March 17:** 5.2°C – 16.7°C, no rain
- **March

# 10. Demo: Full Chain — Three Skills

This query requires all three skills — `country`, `weather`, and `holiday` — working together. The Advisor calls them **in parallel**, then synthesizes a complete travel brief.

In [16]:
response = await run_query(
    "I'm planning a trip to Japan. Give me a complete travel brief — "
    "country info, Tokyo weather, and upcoming public holidays for 2026."
)
print(response)

Query: I'm planning a trip to Japan. Give me a complete travel brief — country info, Tokyo weather, and upcoming public holidays for 2026.
### Travel Advisory: Trip to Japan

#### Country Overview
- **Official Name:** Japan
- **Capital:** Tokyo
- **Region:** Asia / Eastern Asia
- **Population:** Approximately 123.2 million
- **Language:** Japanese
- **Currency:** Japanese yen (¥)
- **Timezone:** UTC+09:00
- **Driving:** Left side of the road

#### Weather in Tokyo (March 2026)
During your visit in March 2026, expect mild temperatures in Tokyo. Here’s a brief forecast for the week of March 15-21:

- **March 15:** 4.9°C – 13.9°C, no rain
- **March 16:** 6.6°C – 12.7°C, no rain
- **March 17:** 4.1°C – 14.1°C, no rain
- **March 18:** 5.2°C – 14.7°C, no rain
- **March 19:** 8.7°C – 15.0°C, light rain (1.1 mm)
- **March 20:** 5.8°C – 18.0°C, light rain (0.4 mm)
- **March 21:** 3.9°C – 15.8°C, no rain

**Current Conditions:** 13.0°C, 45% humidity, wind speed at 3.5 km/h.

#### Upcoming Public

# 11. Demo: Open-Ended Reasoning

This is the most interesting case. The Advisor must decide **on its own** which skills to call and with what queries. To compare Brazil vs Thailand, it needs to call `country` and `weather` **twice each** — and the LLM must choose representative cities to check weather for.

In [17]:
response = await run_query(
    "I'm deciding between Brazil and Thailand for a beach holiday. "
    "Compare both countries and their weather to help me decide."
)
print(response)

Query: I'm deciding between Brazil and Thailand for a beach holiday. Compare both countries and their weather to help me decide.
### Travel Advisory: Beach Holiday Comparison - Brazil vs. Thailand

When choosing between Brazil and Thailand for a beach holiday, both destinations offer stunning coastlines, vibrant cultures, and unique experiences. Here’s a comparison to help you decide:

#### Weather Overview

**Brazil (Rio de Janeiro)**
- **Current Temperature:** 20.6°C
- **Humidity:** 100%
- **7-Day Forecast:**
  - Temperatures range from 21.0°C to 33.3°C.
  - Minimal rain expected, with the highest forecasted at 7.5 mm on March 21.
- **Best Time to Visit:** Generally, the summer months (December to March) are warm and ideal for beach activities.

**Thailand (Phuket)**
- **Current Temperature:** 30.7°C
- **Humidity:** 61%
- **7-Day Forecast:**
  - Temperatures range from 27.0°C to 31.5°C.
  - Light rain expected, with the highest at 3.2 mm on March 18.
- **Best Time to Visit:** Novembe

# 12. Cleanup

In [ ]:
# Terminate both agent servers
data_proc.terminate()
advisor_proc.terminate()
data_proc.wait()
advisor_proc.wait()
print("Both agent servers terminated.")

# 13. Conclusion

In this notebook, we built a **multi-agent travel advisory system** using the Agent2Agent protocol:

- **One Data Agent** with three skills (`weather`, `country`, `holiday`) wrapping free public APIs
- **One Advisor Agent** that uses an LLM to plan, delegate via A2A, and synthesize
- **A thin client** in this notebook that sends JSON-RPC messages and displays the advisory

### Key Takeaways

1. **Agent Cards are the A2A discovery mechanism.** Every agent publishes a card at `/.well-known/agent.json` describing its name, capabilities, and skills — like an OpenAPI spec for agents.

2. **Skills describe what an agent can do.** The Advisor reads skills from the Agent Card at runtime and uses the LLM to decide which to invoke.

3. **JSON-RPC 2.0 is the wire protocol.** Every A2A message is a `message/send` JSON-RPC call. This is standard, language-agnostic, and easy to debug.

4. **AgentExecutor is the server pattern.** Implement `execute()` to handle requests and enqueue results — the SDK handles routing, serialization, and task management.

5. **A2A is the Linux Foundation standard.** Backed by Google, AWS, Microsoft, Salesforce, SAP, and 50+ partners — the broadest industry adoption of any agent-to-agent protocol.

### MCP vs ACP vs A2A — When to Use Which

| Protocol | Best For |
|----------|----------|
| **MCP** | Giving one LLM access to tools and data sources (vertical integration) |
| **ACP** | Quick Python-native agent composition with minimal boilerplate |
| **A2A** | Industry-standard agent interoperability across languages, vendors, and clouds |

All three are complementary. Use **MCP** inside agents for tool access, **A2A** between agents for interoperability, and **ACP** when you need a lightweight Python-first alternative.

### Next Steps

- Add more skills to the data agent (flight prices, exchange rates, points of interest) — the Advisor discovers them automatically via the Agent Card
- Deploy agents on separate machines to demonstrate true distributed agent networks
- Add streaming responses using A2A's `message/stream` for real-time output
- Build multi-hop chains where agents call other agents
- Combine MCP and A2A: have data agents use MCP tools internally while exposing themselves via A2A